Este notebook procesa artículos antiguos en texto OCR para encontrar datos sobre secciones transversales y energías relacionadas con neutrones. Al final organiza los resultados por especie y por elemento.

In [4]:
import re, zipfile
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option('display.max_colwidth', 110)
pd.set_option('display.width', 220)
pd.set_option('display.max_rows', 200)


ZIP_PATH = next((p for p in ['dataset.zip', '/mnt/user-data/uploads/dataset.zip'] if Path(p).exists()), None)
assert ZIP_PATH, 'No encuentro dataset.zip: ponlo junto al notebook o ajusta ZIP_PATH'

raw_docs = {}
with zipfile.ZipFile(ZIP_PATH) as z:
    for n in sorted(z.namelist()):
        if n.endswith('.txt'):
            raw_docs[Path(n).stem] = z.read(n).decode('utf-8', errors='replace')
print(len(raw_docs), 'documentos cargados')

41 documentos cargados


Esta celda limpia errores comunes del OCR y de las fórmulas para que los patrones puedan reconocer mejor los símbolos, números y unidades.

In [5]:
ELEMENTS = {'H':1,'He':2,'Li':3,'Be':4,'B':5,'C':6,'N':7,'O':8,'F':9,'Na':11,'Mg':12,'Al':13,'Si':14,'P':15,
 'S':16,'Cl':17,'K':19,'Ca':20,'V':23,'Cr':24,'Mn':25,'Fe':26,'Co':27,'Ni':28,'Cu':29,'Zn':30,'Br':35,'Mo':42,
 'Rh':45,'Ag':47,'Cd':48,'In':49,'Sb':51,'I':53,'La':57,'Sm':62,'Eu':63,'Gd':64,'Dy':66,'Ta':73,'W':74,'Ir':77,
 'Pt':78,'Au':79,'Hg':80,'Pb':82,'Bi':83,'Th':90,'U':92,'Pu':94}

NAMES = {'hydrogen':'H','helium':'He','lithium':'Li','beryllium':'Be','boron':'B','carbon':'C','nitrogen':'N',
 'oxygen':'O','fluorine':'F','sodium':'Na','magnesium':'Mg','aluminum':'Al','aluminium':'Al','silicon':'Si',
 'phosphorus':'P','sulfur':'S','sulphur':'S','chlorine':'Cl','potassium':'K','calcium':'Ca','vanadium':'V',
 'chromium':'Cr','manganese':'Mn','iron':'Fe','cobalt':'Co','nickel':'Ni','copper':'Cu','zinc':'Zn','bromine':'Br',
 'molybdenum':'Mo','rhodium':'Rh','silver':'Ag','cadmium':'Cd','indium':'In','antimony':'Sb','iodine':'I',
 'lanthanum':'La','samarium':'Sm','europium':'Eu','gadolinium':'Gd','dysprosium':'Dy','tantalum':'Ta',
 'tungsten':'W','iridium':'Ir','platinum':'Pt','gold':'Au','mercury':'Hg','bismuth':'Bi','thorium':'Th',
 'uranium':'U','plutonium':'Pu','lead':'Pb'}
SYM2NAME = {}
for k, v in NAMES.items():
    SYM2NAME.setdefault(v, k)

def _ce(m):
    s = m.group(1).strip()
    return SYM2NAME[s] if s in SYM2NAME else s

NEST = r'((?:[^{}]|\{[^{}]*\})*)'

def clean_ocr(t: str) -> str:
    t = t.replace('\u2014', '-').replace('\u2212', '-').replace('\u00b1', '±')
    t = t.replace('γ', 'gamma').replace('α', 'alpha').replace('β', 'beta').replace('σ', 'sigma')
    t = re.sub(r'\\ce\{\^\{(\d+)\}\s*([A-Z][a-z]?)\s*\}', r'\2\1', t)
    t = re.sub(r'\\ce\{' + NEST + r'\}', _ce, t)
    t = re.sub(r'\\text\{' + NEST + r'\}', r'\1', t)
    t = re.sub(r'\\(?:left|right|mathrm|rm)\b', '', t)
    t = t.replace('$', ' ')
    t = re.sub(r'([A-Z][a-z]?)\s?\^\{(\d{1,3})\}', r'\1\2', t)
    t = re.sub(r'\^\{(\d{1,3})\}\s?([A-Z][a-z]?)\b', r'\2\1', t)
    t = re.sub(r'\(\s*n\s*,\s*\\rho\s*\)', '(n, p)', t)
    t = re.sub(r'\(\s*y\s*,', '(gamma,', t)
    t = t.replace('\\pm', '±').replace('\\times', 'x').replace('\\sim', '~')
    t = re.sub(r'\^\{(-?\s?\d+)[}\]]', r'^\1', t)
    t = re.sub(r'\^\{([^{}]*)\}', r'^\1', t)
    t = re.sub(r'_\{([^{}]*)\}', r'_\1', t)
    t = re.sub(r'\{(10|cm)\}', r'\1', t)
    t = re.sub(r'\\([A-Za-z]+)', r'\1', t)
    t = t.replace('[EQUATION]', ' ')
    t = re.sub(r'(?<=\d)\.\s+(?=\d)', '.', t)
    t = re.sub(r'(?<=\d),\s(?=\d{3}\b)', '', t)
    return re.sub(r'\s+', ' ', t).strip()

def split_sentences(t):
    return [s for s in re.split(r'(?<=[a-z0-9\)%])\.\s+(?=[A-Z])', t) if s.strip()]

docs = {k: clean_ocr(v) for k, v in raw_docs.items()}
sents = {k: split_sentences(v) for k, v in docs.items()}
print('oraciones totales:', sum(len(v) for v in sents.values()))
print(docs['1946_Rainwater'][600:1100])

oraciones totales: 3138
btained by using an extensive B4, carbon and cadmium collimating system. Experiments were conducted principally at a 5.4meter source detector distance. A special selector system counts all the neutrons detected and also selectively counts those detected in an adjustable timed interval after the cyclotron burst. The slow neutronenergy distributions from paraffin "source" slabs were shown to be of a modified Maxwellian form with an asymmetrical high energy "tail. " Data for the resonance absorptio


Aquí se definen los materiales y las reglas para detectar isótopos, elementos, compuestos y otros materiales mencionados en el texto.

In [6]:
MATERIALS = {
    'paraffin': r'paraffin', 'polyethylene': r'polyethylene|polythene', 'anthracene': r'anthracene',
    'stilbene': r'stilbene', 'emulsion': r'(?:nuclear|photographic)?\s?emulsions?',
    'NaI': r'sodium iodide|NaI\b', 'LiI': r'lithium iodide|LiI\b', 'ZnS': r'zinc sulfide|zinc sulphide|ZnS\b', 'lucite': r'lucite',
}
LEAD_NOUNS = r'(?=\s(?:foil|foils|absorber|absorbers|plate|plates|sheet|sheets|target|targets|shield|shielding|bricks?|slab|slabs|cylinders?|nitrate|glass|filter|filters)\b)'

ISO_RE   = re.compile(r'(?<![A-Za-z0-9])([A-Z][a-z]?)-?(\d{1,3})(?![\d+])')
BF3_RE   = re.compile(r'\bBF3\b|boron trifluoride', re.I)
NAME_RE  = re.compile(r'\b(' + '|'.join(NAMES) + r'|deuterium|tritium)\b', re.I)
MAT_RE   = {k: re.compile(r'\b(?:' + v + r')', re.I) for k, v in MATERIALS.items()}

def plausible(Z, A):
    return Z <= A <= 2.7 * Z + 3 and A >= 1.9 * Z - 1

def find_species(s):

    out = []
    for m in ISO_RE.finditer(s):
        el, A = m.group(1), int(m.group(2))
        if el in ELEMENTS and plausible(ELEMENTS[el], A):
            out.append((m.start(), f'{el}-{A}', el, 'isotope', A))
    for m in BF3_RE.finditer(s):
        out.append((m.start(), 'BF3', 'B', 'compound', np.nan))
    for m in NAME_RE.finditer(s):
        name = m.group(1).lower()
        if name == 'deuterium':
            out.append((m.start(), 'H-2', 'H', 'isotope', 2)); continue
        if name == 'tritium':
            out.append((m.start(), 'H-3', 'H', 'isotope', 3)); continue
        if name == 'lead' and not re.match(LEAD_NOUNS, s[m.end():], re.I):
            continue
        if re.match(r'\s(?:iodide|sulfide|sulphide|trifluoride)', s[m.end():], re.I):
            continue
        sym = NAMES[name]
        out.append((m.start(), sym, sym, 'element', np.nan))
    for key, rx in MAT_RE.items():
        for m in rx.finditer(s):
            out.append((m.start(), key, None, 'material', np.nan))
    return sorted(out)

demo = find_species(docs['1950_Hibdon'][:1500])
print(demo[:8])

[(336, 'Co', 'Co', 'element', nan), (347, 'Mn', 'Mn', 'element', nan), (372, 'Co', 'Co', 'element', nan), (383, 'Mn', 'Mn', 'element', nan), (1201, 'Co', 'Co', 'element', nan), (1212, 'Mn', 'Mn', 'element', nan), (1292, 'Co', 'Co', 'element', nan), (1303, 'Mn', 'Mn', 'element', nan)]


Aquí se preparan los patrones que buscan secciones transversales y energías, y se convierten los valores a unidades comparables.

In [7]:
NUM = r'\d+(?:\.\d+)?'


XS_BARN = re.compile(rf'\(?\s*(?P<v>{NUM})\s*(?:±\s*(?P<e>{NUM})\s*)?\)?\s*(?P<u>milli)?barns?\b', re.I)
XS_E24  = re.compile(rf'\(?\s*(?P<v>{NUM})\s*(?:±\s*(?P<e>{NUM})\s*)?\)?\s*[xX×]\s*10\^?\s?(?P<x>-?\s?\d{{2}})\s*cm\^?2?')
XS_KW   = re.compile(r'cross.?section|sigma|absorption|capture|scatter|fission|activation', re.I)
XS_TYPES = [('capture', r'capture'), ('absorption', r'absorption|absorb'), ('scattering', r'scatter'),
            ('fission', r'fission'), ('total', r'\btotal\b'), ('activation', r'activation')]


EN = re.compile(rf'(?P<v>{NUM})\s*(?:(?:to|-)\s*(?P<v2>{NUM})\s*|±\s*(?P<e>{NUM})\s*)?(?P<u>[MmKkGg]?[Ee][Vv])\b')
UNIT_MEV = {'ev': 1e-6, 'kev': 1e-3, 'mev': 1.0, 'gev': 1e3}
PARTICLES = {
    'alpha':     r'\balpha[- ]?(?:particles?|rays?|s)?\b|\balfa\b|\bdisintegration alphas?\b',
    'proton':    r'\bprotons?\b',
    'triton':    r'\btritons?\b',
    'deuteron':  r'\bdeuterons?\b',
    'recoil':    r'\brecoil\w*',
    'fragment':  r'\bfragments?\b',
    'beta':      r'\bbeta[- ]?(?:rays?|particles?)?\b|\bbetarays?\b|\belectrons?\b',
    'gamma':     r'\bgamma[- ]?rays?\b|\bgamma\b|\bgammarays?\b|\byrays?\b',
    'neutron':   r'\bneutrons?\b|\bphotoneutrons?\b',
}
PART_RE = {k: re.compile(v, re.I) for k, v in PARTICLES.items()}
OUTGOING = {'alpha', 'proton', 'triton', 'deuteron', 'recoil', 'fragment', 'beta', 'gamma'}
CHARGED = {'alpha', 'triton', 'proton', 'recoil', 'fragment'}
E_OUT_MAX_MEV = 30
BEAM_RE = re.compile(r'cyclotron|betatron|synchrotron|bombard|accelerat|(?:deuteron|proton|electron|alpha) beam', re.I)
BEAM_PARTS = {'deuteron', 'proton', 'alpha', 'beta', 'gamma'}
Q_RE = re.compile(r'\bQ[- ]?values?\b|disintegration energy|energy release', re.I)

REACTION = re.compile(r'(?P<tgt>[A-Z][a-z]?\d{0,3})\s?\(\s?(?P<in>n|p|d|alpha|a|gamma|x)\s?,\s?(?P<out>alpha|a|p|n|d|t|gamma|2n|f|x)\s?\)\s?(?P<prod>[A-Z][a-z]?\d{0,3})?')

REGIME = [('thermal/slow', r'thermal|slow neutron|slow-neutron'), ('resonance', r'resonance'), ('fast', r'\bfast\b|mev neutron')]

def first_match(pairs, s, default=None):
    for label, pat in pairs:
        if re.search(pat, s, re.I):
            return label
    return default

def as_num(x):
    return float(x) if x is not None else np.nan

def extract_xs(s):
    hits = []
    if not XS_KW.search(s):
        return hits
    for m in XS_BARN.finditer(s):
        f = 1e-3 if m.group('u') else 1.0
        hits.append(dict(metric='sigma_barn', value=as_num(m.group('v')) * f, err=as_num(m.group('e')) * f,
                         raw=m.group(0).strip(), pos=m.start()))
    for m in XS_E24.finditer(s):
        exp = int(m.group('x').replace(' ', ''))
        exp = -abs(exp)
        if not 22 <= abs(exp) <= 28:
            continue
        f = 10.0 ** (exp + 24)
        hits.append(dict(metric='sigma_barn', value=as_num(m.group('v')) * f, err=as_num(m.group('e')) * f,
                         raw=m.group(0).strip(), pos=m.start()))
    for h in hits:
        h['xs_type'] = first_match(XS_TYPES, s, 'unspecified')
        h['regime'] = first_match(REGIME, s)
    return hits

def extract_energy(s):
    hits = []
    parts = [(m.start(), k) for k, rx in PART_RE.items() for m in rx.finditer(s)]
    qs = [m.start() for m in Q_RE.finditer(s)]
    beam_ctx = bool(BEAM_RE.search(s))
    for m in EN.finditer(s):
        u = m.group('u').lower()
        v = as_num(m.group('v'))
        if m.group('v2'):
            v = (v + float(m.group('v2'))) / 2
        val = v * UNIT_MEV[u]
        near = [(abs(p - m.start()), k) for p, k in parts if abs(p - m.start()) <= 120]
        particle = min(near)[1] if near else None
        is_q = any(0 <= m.start() - q <= 40 for q in qs)
        if is_q:
            role = 'q_value'
        elif particle in BEAM_PARTS and beam_ctx:
            role = 'beam'
        elif particle in OUTGOING and u in ('kev', 'mev'):
            role = 'outgoing'
        elif particle == 'neutron' or u == 'ev':
            role = 'incident'
        else:
            role = 'unknown'
        hits.append(dict(metric='E_out_MeV', value=val,
                         err=as_num(m.group('e')) * UNIT_MEV[u] if m.group('e') else np.nan,
                         raw=m.group(0).strip(), pos=m.start(), particle=particle, role=role,
                         e_range=bool(m.group('v2'))))
    return hits

def sample(pat_fn, doc, n=4):
    out = []
    for s in sents[doc]:
        out += [(h['value'], h['raw']) for h in pat_fn(s)]
    return out[:n]

print('sigma  :', sample(extract_xs, '1946_Rainwater'))
print('energia:', sample(extract_energy, '1946_Bennett'))

sigma  : [(7800.0, '( 7800 ± 800 ) x 10^-24 cm^2')]
energia: [(4.9, '4.9Mev'), (1.35, '1.35 Mev'), (17.69, '17.69 Mev'), (1.35, '1.35 Mev')]


Esta parte recorre los documentos, guarda las reacciones detectadas y relaciona cada valor encontrado con la especie mencionada más cerca en la oración.

In [8]:
rows, react_rows = [], []
for doc, ss in sents.items():
    year = int(doc[:4])
    recent = []
    for i, s in enumerate(ss):
        sp = find_species(s)
        for m in REACTION.finditer(s):
            react_rows.append(dict(doc=doc, year=year, sent_id=i, target=m.group('tgt'), incident=m.group('in'),
                                   outgoing=m.group('out'), product=m.group('prod'), raw=m.group(0)))
        hits = extract_xs(s) + extract_energy(s)
        rx = next(REACTION.finditer(s), None)
        for h in hits:
            inherited = False
            if sp:
                pos, key, el, typ, A = min(sp, key=lambda t: abs(t[0] - h['pos']))
            elif recent:
                pos, key, el, typ, A = recent[-1]; inherited = True
            else:
                key = el = typ = None; A = np.nan
            rows.append(dict(doc=doc, year=year, sent_id=i, especie=key, elemento=el, tipo=typ, A=A,
                             metric=h['metric'], value=h['value'], err=h['err'], raw=h['raw'],
                             xs_type=h.get('xs_type'), regime=h.get('regime'), particle=h.get('particle'),
                             role=h.get('role'), e_range=h.get('e_range'),
                             reaccion=rx.group(0) if rx else None, inherited=inherited,
                             ctx_neutron=bool(re.search(r'neutron', s, re.I)),
                             ctx_gamma=bool(re.search(r'gamma|x-?ray|yray', s, re.I)),
                             sentence=s[:350]))
        if sp:
            recent = (recent + [sorted(sp)[-1]])[-2:]
        elif recent and i % 3 == 2:
            recent = recent[-1:]

obs = pd.DataFrame(rows)
reacciones = pd.DataFrame(react_rows)
print('observaciones:', len(obs), '| reacciones:', len(reacciones))
print(obs.groupby('metric').agg(n=('value', 'size'), docs=('doc', 'nunique'), sin_especie=('especie', lambda x: x.isna().sum())))

observaciones: 470 | reacciones: 28
              n  docs  sin_especie
metric                            
E_out_MeV   448    35           25
sigma_barn   22     7            0


In [9]:
obs.groupby(['metric', 'doc']).size().unstack(0, fill_value=0).assign(total=lambda d: d.sum(axis=1)).sort_values('total', ascending=False).head(25)

metric,E_out_MeV,sigma_barn,total
doc,,,
1948_Seidl,61,1,62
1940_Haxby,52,0,52
1942_Bacher,32,7,39
1949_McDaniel,37,0,37
1946_Bennett,35,0,35
1940_Herbert,21,9,30
1950_Hibdon,23,2,25
1950_Roger,23,0,23
1949_Price,20,0,20


Estas tablas permiten revisar ejemplos de los datos extraídos y detectar resultados que puedan venir de errores del OCR.

In [10]:
cols = ['doc', 'especie', 'value', 'err', 'raw', 'xs_type', 'regime', 'inherited']
print('--- SECCIÓN TRANSVERSAL (barns) ---')
display(obs[obs.metric == 'sigma_barn'][cols].sort_values('value', ascending=False).head(25))

--- SECCIÓN TRANSVERSAL (barns) ---


,doc,especie,value,err,raw,xs_type,regime,inherited
80,1940_Herbert,U,10000.000,NaN,10000 X 10^-24 cm^2,unspecified,NaN,True
76,1940_Herbert,B,9200.000,NaN,9200 X 10^24 cm^2,unspecified,resonance,True
157,1946_Rainwater,B,7800.000,800.0,( 7800 ± 800 ) x 10^-24 cm^2,absorption,thermal/slow,False
63,1940_Herbert,B,5000.000,NaN,5000X10^-24 cm^2,unspecified,NaN,True
61,1940_Herbert,B,4600.000,NaN,4600 X10^-24 cm^2,unspecified,NaN,True
53,1940_Herbert,Cd,4600.000,NaN,4600 x 10^-24 cm^2,absorption,thermal/slow,False
60,1940_Herbert,B,4600.000,NaN,4600X 10^-24 cm^2,absorption,resonance,False
74,1940_Herbert,B,4600.000,NaN,4600 X 10^-24 cm^2,unspecified,NaN,True
62,1940_Herbert,B,1200.000,NaN,1200X10^-24 cm^2,unspecified,NaN,True
110,1942_Bacher,B,708.000,NaN,708X10^-24 cm^2,absorption,thermal/slow,True


In [11]:
print('--- ENERGÍA (MeV) por rol ---')
print(obs[obs.metric == 'E_out_MeV'].role.value_counts().to_string())
display(obs[(obs.metric == 'E_out_MeV') & (obs.role.isin(['outgoing', 'q_value']))]
        [['doc', 'especie', 'value', 'err', 'particle', 'role', 'raw', 'reaccion']].head(40))

--- ENERGÍA (MeV) por rol ---
role
incident    294
unknown      99
outgoing     37
beam         17
q_value       1


,doc,especie,value,err,particle,role,raw,reaccion
1,1940_Haxby,paraffin,0.862,NaN,gamma,outgoing,0.862MeV,NaN
12,1940_Haxby,F,0.862,NaN,gamma,outgoing,0.862Mev,NaN
13,1940_Haxby,F,0.862,NaN,gamma,outgoing,0.862Mev,NaN
14,1940_Haxby,F,2.586,NaN,gamma,outgoing,2.586 Mev,NaN
18,1940_Haxby,paraffin,0.862,NaN,gamma,outgoing,0.862 Mev,NaN
19,1940_Haxby,paraffin,1.724,NaN,gamma,outgoing,1.724 Mev,NaN
20,1940_Haxby,paraffin,2.856,NaN,gamma,outgoing,2.856 Mev,NaN
32,1940_Haxby,paraffin,0.870,NaN,gamma,outgoing,0.87 Mev,NaN
33,1940_Haxby,paraffin,0.450,NaN,gamma,outgoing,0.45 Mev,NaN
34,1940_Haxby,paraffin,0.420,NaN,gamma,outgoing,0.42 Mev,NaN


In [12]:
print('--- Reacciones (n,x) detectadas ---')
display(reacciones[reacciones.incident == 'n'].groupby(['target', 'outgoing']).size().sort_values(ascending=False).head(20).rename('n').reset_index())

--- Reacciones (n,x) detectadas ---


,target,outgoing,n
0,C12,2n,7
1,H,gamma,2
2,B,a,1
3,Ag107,2n,1
4,C,2n,1
5,Li6,a,1
6,Mg24,p,1
7,N,p,1
8,P31,p,1
9,S32,p,1


Esta celda resume los datos por especie y por elemento usando medianas, rangos y cantidad de observaciones. También indica qué grupos tienen ambas métricas.

In [13]:
def build_vectors(df, key):
    d = df[df[key].notna()].copy()
    sig = d[d.metric == 'sigma_barn']
    eng = d[(d.metric == 'E_out_MeV') & d.role.isin(['outgoing', 'q_value']) & ~d.inherited & (d.value <= E_OUT_MAX_MEV)
            & (d.particle.isin(CHARGED) | (d.role == 'q_value'))]
    def stats(x, name):
        return x.groupby(key).agg(**{f'{name}': ('value', 'median'), f'{name}_min': ('value', 'min'),
                                     f'{name}_max': ('value', 'max'), f'n_{name}': ('value', 'size')})
    out = stats(sig, 'sigma_b').join(stats(eng, 'E_out_MeV'), how='outer')
    docs_ = d.groupby(key)['doc'].agg(lambda x: ', '.join(sorted(set(x))))
    out = out.join(docs_.rename('docs')).reset_index()
    for c in [c for c in out.columns if c.startswith('n_')]:
        out[c] = out[c].fillna(0).astype(int)
    out.insert(out.columns.get_loc('sigma_b') + 1, 'log10_sigma', np.log10(out['sigma_b'].where(out['sigma_b'] > 0)))
    out['n_metricas'] = out[['sigma_b', 'E_out_MeV']].notna().sum(axis=1)
    out['completo'] = out.n_metricas == 2
    return out.sort_values(['n_metricas', 'n_sigma_b'], ascending=False).reset_index(drop=True)

vec_especie = build_vectors(obs, 'especie')
vec_elemento = build_vectors(obs.assign(grupo=obs.elemento.fillna(obs.especie)), 'grupo').rename(columns={'grupo': 'elemento'})

print('Especies con ≥1 métrica:', len(vec_especie), '| con las 2:', int(vec_especie.completo.sum()))
print('Elementos con ≥1 métrica:', len(vec_elemento), '| con las 2:', int(vec_elemento.completo.sum()))
display(vec_especie[['especie', 'sigma_b', 'log10_sigma', 'E_out_MeV', 'n_sigma_b', 'n_E_out_MeV', 'n_metricas']].head(30))

Especies con ≥1 métrica: 12 | con las 2: 1
Elementos con ≥1 métrica: 11 | con las 2: 2


,especie,sigma_b,log10_sigma,E_out_MeV,n_sigma_b,n_E_out_MeV,n_metricas
0,C,4.000,0.602060,7.000,2,1,2
1,B,4600.000,3.662758,NaN,10,0,1
2,Cd,708.000,2.850033,NaN,3,0,1
3,paraffin,552.000,2.741939,NaN,3,0,1
4,Al,0.025,-1.602060,NaN,1,0,1
5,Co,8.000,0.903090,NaN,1,0,1
6,H-3,0.100,-1.000000,NaN,1,0,1
7,U,10000.000,4.000000,NaN,1,0,1
8,B-10,NaN,NaN,4.915,0,2,1
9,Be-8,NaN,NaN,2.800,0,1,1


In [14]:
display(vec_elemento[['elemento', 'sigma_b', 'log10_sigma', 'E_out_MeV', 'n_sigma_b', 'n_E_out_MeV', 'n_metricas']].head(30))

,elemento,sigma_b,log10_sigma,E_out_MeV,n_sigma_b,n_E_out_MeV,n_metricas
0,B,4600.000,3.662758,4.915,10,2,2
1,C,4.000,0.602060,7.000,2,1,2
2,Cd,708.000,2.850033,NaN,3,0,1
3,paraffin,552.000,2.741939,NaN,3,0,1
4,Al,0.025,-1.602060,NaN,1,0,1
5,Co,8.000,0.903090,NaN,1,0,1
6,H,0.100,-1.000000,NaN,1,0,1
7,U,10000.000,4.000000,NaN,1,0,1
8,Be,NaN,NaN,2.800,0,1,1
9,He,NaN,NaN,1.000,0,1,1


Al final se guardan las observaciones, reacciones y resúmenes en archivos CSV para usarlos en el siguiente análisis.

In [15]:
obs.to_csv('observaciones.csv', index=False, encoding='utf-8-sig')
reacciones.to_csv('reacciones.csv', index=False, encoding='utf-8-sig')
vec_especie.to_csv('vectores_por_especie.csv', index=False, encoding='utf-8-sig')
vec_elemento.to_csv('vectores_por_elemento.csv', index=False, encoding='utf-8-sig')
print('Listo: observaciones.csv, reacciones.csv, vectores_por_especie.csv, vectores_por_elemento.csv')

Listo: observaciones.csv, reacciones.csv, vectores_por_especie.csv, vectores_por_elemento.csv
